# 02 Preprocessing


## CRISP-DM Stage
Data Preparation.

## Research Context
Phase 7A prepares the completed Google Play raw review dataset for downstream IndoBERT sentiment modeling and SVM aspect classification. This phase does not train models, fit TF-IDF, download model artifacts, or split train/validation/test data.

## Preprocessing Objective
The objective is to preserve the original `initial_sentiment`, create a conservative corrected `final_sentiment`, and produce two text representations: `text_indobert` for IndoBERT and `text_svm` for later SVM feature extraction.

## Input Dataset
Primary input:
- `datasets/raw/reviews_raw_labeled.csv`

Expected columns include `source`, `app_id`, `external_id`, `author_name`, `rating`, `content`, `likes`, `app_version`, `reviewed_at`, `scraped_at`, `sort_method`, and `initial_sentiment`. Optional columns are handled safely by the scripts.

## Initial Sentiment Label Review
`initial_sentiment` is derived from star rating in the acquisition phase and must remain unchanged. It is retained as an audit column for comparing rating-based labels against the keyword-corrected `final_sentiment`.

## Keyword-based Relabeling Strategy
Relabeling is intentionally conservative. Neutral rows may change to `Negative` or `Positive` only when a one-direction strong keyword signal appears. Mixed signals are not relabeled and are marked as audit candidates through `relabel_reason`. Rating 1, 2, 4, and 5 rows are not aggressively relabeled; strong contradictions are documented for audit only.

## IndoBERT Preprocessing Strategy
`text_indobert` uses conservative text cleaning: Unicode normalization, URL removal, HTML tag/entity cleanup, zero-width space cleanup, and whitespace normalization. It preserves sentence structure, negation words, punctuation where possible, and emoji where possible. It does not apply stemming or stopword removal.

## SVM Preprocessing Strategy
`text_svm` uses stronger normalization suitable for later feature extraction: lowercasing, URL removal, HTML cleanup, conservative repeated-character normalization, punctuation cleanup, and simple emoji token handling. It does not fit TF-IDF, create vectorizer artifacts, stem text, or split the dataset.

## Negation Preservation Note
Negation words such as `tidak`, `bukan`, `belum`, `jangan`, `ga`, `gak`, and `nggak` must be preserved because they can invert sentiment meaning. Neither preprocessing script removes stopwords aggressively.

## Emoji Handling Note
IndoBERT preprocessing avoids destructive emoji removal. SVM preprocessing converts emoji-like Unicode ranges into a simple `emoji` token so the signal is not silently discarded before feature extraction.

## Data Validation Plan
Validation checks include total rows, label distribution before and after relabeling, changed label count, audit candidate count, rating-3 changed count when `rating` exists, missing text counts, empty processed text counts, and text length statistics before and after cleaning.

## EDA Before vs After Plan
Preprocessing EDA compares label distributions before and after keyword relabeling and compares raw, IndoBERT, and SVM text length statistics. Figures are exported to `docs/figures/02_preprocessing/`.

## Aspect Taxonomy Requirement
Aspect taxonomy is needed because SVM will later classify review themes, while AHP and Fuzzy AHP require validated actionable criteria for priority ranking. Phase 7B derives candidate aspects from exploratory review analysis only; it does not establish final expert judgement.

## Negative and Neutral Review Focus
Candidate term extraction prioritizes `Negative` and `Neutral` reviews because complaint and ambiguity patterns are most useful for identifying improvement themes. Positive reviews remain part of weak labeling output but are not the main source for deriving draft criteria.

## Draft Aspect Taxonomy
Initial candidate aspects:
- Performance & Stability
- Ads Experience
- Subscription & Pricing
- Features & Content
- Audio Quality
- UI/UX
- Account/Login
- General

`General` is a fallback class for weak labeling only and must not be used as an AHP/Fuzzy AHP criterion later.

## Keyword-based Weak Aspect Labeling Strategy
The weak labeling script scores keyword matches for each draft aspect and assigns the aspect with the highest score. If no aspect keyword is matched, the row receives `General`. These labels are preparation for SVM aspect classification experiments and must not be treated as expert-validated ground truth.

## Expert Judgement Requirement
Candidate aspects are derived from review text exploration and keyword rules. Before AHP or Fuzzy AHP weighting, actionable aspects must be reviewed and validated through expert judgement. The final decision criteria should exclude `General` and include only validated, actionable improvement areas.

## Phase 7C General Fallback Refinement
Phase 7B produced a high `General` fallback rate: 46,704 of 97,782 rows, or 47.7634%. This is problematic for SVM aspect classifier training because too many fallback labels reduce aspect specificity and can make the classifier learn a vague residual class instead of actionable review themes.

## General Fallback Analysis Strategy
Phase 7C analyzes terms and bigrams from `General` fallback rows, with additional focus on `Negative` and `Neutral` fallback reviews. The result is used to refine keyword coverage while preserving the warning that weak labels are exploratory and not expert-validated ground truth.

## Refined Aspect Distribution
| Aspect label | Rows |
| --- | ---: |
| Performance & Stability | 2,600 |
| Ads Experience | 16,282 |
| Subscription & Pricing | 10,663 |
| Features & Content | 21,703 |
| Audio Quality | 442 |
| UI/UX | 351 |
| Account/Login | 1,474 |
| General | 44,267 |

The refined `General` fallback rate is 45.2711%. The reduction is useful but still limited, so `General` remains a quality-control concern before SVM modeling.

## Aspect Confidence Metadata
The refined weak labels include `aspect_keyword_score`, `aspect_keywords_matched`, and `aspect_label_confidence`.

| Confidence | Rows |
| --- | ---: |
| none | 44,267 |
| low | 36,532 |
| medium | 9,811 |
| high | 7,172 |

Later SVM training should prioritize high and medium confidence labels or use a balanced subset with explicit audit rules. Low confidence labels should be treated cautiously.

## Processed Dataset Outputs
Generated local processed datasets:
- `datasets/processed/reviews_relabelled.csv`
- `datasets/processed/reviews_preprocessed_indobert.csv`
- `datasets/processed/reviews_preprocessed_svm.csv`
- `datasets/processed/reviews_final.csv`
- `datasets/processed/reviews_with_aspect_labels.csv`
- `datasets/processed/reviews_with_aspect_labels_refined.csv`

These CSV files are generated artifacts and must remain ignored by Git.

## Frontend Metrics Outputs
Aggregate preprocessing metrics are exported to `datasets/outputs/eda/02_preprocessing/` for later dashboard visualization:
- `label_distribution_before_relabeling.csv`
- `label_distribution_before_relabeling.json`
- `label_distribution_after_relabeling.csv`
- `label_distribution_after_relabeling.json`
- `relabeling_summary.json`
- `text_length_before_after_cleaning.csv`
- `text_length_before_after_cleaning.json`
- `preprocessing_summary.json`
- `aspect_taxonomy_candidate_terms.csv`
- `aspect_taxonomy_derivation_summary.json`
- `aspect_labeling_summary.json`
- `aspect_label_distribution.csv`
- `aspect_label_distribution.json`
- `aspect_by_sentiment_distribution.csv`
- `aspect_by_sentiment_distribution.json`
- `general_fallback_terms.csv`
- `general_fallback_analysis.json`
- `aspect_labeling_refined_summary.json`
- `aspect_label_distribution_refined.csv`
- `aspect_label_distribution_refined.json`
- `aspect_by_sentiment_distribution_refined.csv`
- `aspect_by_sentiment_distribution_refined.json`
- `aspect_label_confidence_distribution.csv`
- `aspect_label_confidence_distribution.json`

## Figure Outputs
Generated Phase 02 figures:
- `docs/figures/02_preprocessing/label_distribution_before_relabeling.png`
- `docs/figures/02_preprocessing/label_distribution_after_relabeling.png`
- `docs/figures/02_preprocessing/relabeling_change_summary.png`
- `docs/figures/02_preprocessing/text_length_before_after_cleaning.png`
- `docs/figures/02_preprocessing/aspect_candidate_terms.png`
- `docs/figures/02_preprocessing/aspect_label_distribution.png`
- `docs/figures/02_preprocessing/aspect_by_sentiment_distribution.png`
- `docs/figures/02_preprocessing/general_fallback_top_terms.png`
- `docs/figures/02_preprocessing/aspect_label_distribution_refined.png`
- `docs/figures/02_preprocessing/aspect_by_sentiment_distribution_refined.png`
- `docs/figures/02_preprocessing/aspect_label_confidence_distribution.png`
- `docs/figures/02_preprocessing/general_rate_before_after.png`

## Reproducible Commands
Run from `ml-service/`:

```bash
uv run python scripts/relabel_by_keywords.py --input ../datasets/raw/reviews_raw_labeled.csv --output ../datasets/processed/reviews_relabelled.csv
uv run python scripts/preprocess_indobert.py --input ../datasets/processed/reviews_relabelled.csv --output ../datasets/processed/reviews_preprocessed_indobert.csv
uv run python scripts/preprocess_svm.py --input ../datasets/processed/reviews_preprocessed_indobert.csv --output ../datasets/processed/reviews_preprocessed_svm.csv
uv run python scripts/derive_aspect_taxonomy.py --input ../datasets/processed/reviews_final.csv --output-summary ../datasets/outputs/eda/02_preprocessing/aspect_taxonomy_derivation_summary.json --output-keywords ../datasets/outputs/eda/02_preprocessing/aspect_taxonomy_candidate_terms.csv
uv run python scripts/label_aspects_by_keywords.py --input ../datasets/processed/reviews_final.csv --output ../datasets/processed/reviews_with_aspect_labels.csv --summary-output ../datasets/outputs/eda/02_preprocessing/aspect_labeling_summary.json
uv run python scripts/derive_aspect_taxonomy.py --input ../datasets/processed/reviews_with_aspect_labels.csv --output-summary ../datasets/outputs/eda/02_preprocessing/general_fallback_analysis.json --output-keywords ../datasets/outputs/eda/02_preprocessing/general_fallback_terms.csv --focus-aspect General
uv run python scripts/label_aspects_by_keywords.py --input ../datasets/processed/reviews_final.csv --output ../datasets/processed/reviews_with_aspect_labels_refined.csv --summary-output ../datasets/outputs/eda/02_preprocessing/aspect_labeling_refined_summary.json
```

## Interpretation Notes
Interpret relabeling and weak aspect labeling results as conservative exploratory preparation, not as ground-truth manual annotation. The refined General rate remains high at 45.2711%, so labels and taxonomy coverage still need expert review before SVM training and before any AHP/Fuzzy AHP criteria validation.

## Limitations
Keyword rules are transparent and reproducible but cannot capture all Indonesian sentiment expressions, sarcasm, context-dependent wording, or overlapping aspect themes. Refinement improves coverage modestly but does not make labels expert validated. Class balancing, dataset splits, tokenization, vectorizer fitting, model training, and expert validation are deferred to later phases.

## Next Step
Continue to `03_indobert_sentiment_modeling.ipynb` for sentiment modeling and `04_svm_aspect_classification.ipynb` for aspect classifier modeling. Final AHP/Fuzzy AHP criteria validation remains a later expert judgement step.